# Complete source and training protocol — 2026-11-08

All Python modules, scripts and tests are merged below in readable functional sections. These are source listings, not execution cells: relative imports and command-line entry points belong to their files. Use the companion quickstart for execution. Rebuild with `python scripts/build_notebooks.py` after changing source files.

## src/solubility_public_data/__init__.py

```python
"""Mini-protein insolubility prediction. All scores use insolubility=1."""
__version__ = "0.1.0"
```

## src/solubility_public_data/cli.py

```python
"""Train/evaluate/deploy without selecting settings on external labels."""
import argparse
from datetime import datetime
import hashlib
import json
from pathlib import Path
import numpy as np
from scipy.stats import spearmanr
from .data import FeatureStore, read_records, subset, SEEDS
from .training import Estimator
from .evaluation import make_splits, validate_splits, choose_threshold, metrics


def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False)+'\n')


def setup(args, training=True):
    records = read_records(args.records, training=training)
    store = FeatureStore(args.features, getattr(args, 'structure', None))
    config = json.loads(Path(args.config).read_text()) if getattr(args, 'config', None) else None
    return records, store, config


def train(args):
    records, store, config = setup(args)
    out = Path(args.output)
    if out.exists() and any(out.iterdir()):
        raise ValueError('Use a new empty output directory for each run')
    out.mkdir(parents=True, exist_ok=True)
    date = args.date
    if args.max_epochs is not None:
        config['max_epochs'] = args.max_epochs
    data = store.align(records, config['architecture'] == 'bilstm', config['profile'] == 'structure')
    y = np.array([r['insoluble'] for r in records], np.float32)
    groups = np.array([r['group_id'] for r in records])
    external = []
    for path in args.external:
        rr = read_records(path)
        if any(r.get('role') != 'external' for r in rr):
            raise ValueError('External panels require role=external')
        if {r['sequence'] for r in rr} & {r['sequence'] for r in records} or {r['group_id'] for r in rr} & set(groups):
            raise ValueError('External exact-sequence/family overlap with training')
        external.append((rr, store.align(rr, config['architecture'] == 'bilstm', config['profile'] == 'structure')))
    frozen = dict(created=date, config=config, seeds=SEEDS, n=len(y), n_insoluble=int(y.sum()), n_groups=len(set(groups)),
        positive_class='insoluble', epochs_capped_for_smoke=args.max_epochs is not None,
        records_sha256=hashlib.sha256(Path(args.records).read_bytes()).hexdigest(),
        threshold_policy='training OOF insoluble recall >= '+str(args.recall),
        external_used_for_selection=False)
    write_json(out/f'protocol_{date}.json', frozen)
    rows, predictions, fits, manifest = [], [], [], {}
    for mode in args.modes:
        if args.splits:
            s = json.loads(Path(args.splits).read_text())
            if s['record_ids'] != [r['record_id'] for r in records]:
                raise ValueError('Split record ID order mismatch')
            splits = s[mode]
        else:
            splits = make_splits(records, mode)
        validate_splits(records, splits, mode); manifest[mode] = splits
        for seed in SEEDS:
            spec = splits[str(seed)]; tr = np.array(spec['train']); te = np.array(spec['test']); folds = np.array(spec['folds'])
            p = np.full(len(y), np.nan)
            for fold in range(4):
                fitting = tr[folds[tr] != fold] if fold < 3 else tr
                testing = tr[folds[tr] == fold] if fold < 3 else te
                estimator = Estimator(config, store, seed+fold*10000, args.device).fit(subset(data, fitting), y[fitting], groups[fitting])
                p[testing] = estimator.predict(subset(data, testing))
                fits.append(dict(mode=mode, outer_seed=seed, fold=fold, fitting_ids=[records[i]['record_id'] for i in fitting], **estimator.metadata))
            threshold = choose_threshold(y[tr], p[tr], args.recall)
            estimator.save(out/f'{mode}_seed{seed}_{date}.pt', threshold)
            for split, ix in [('oof', tr), ('holdout', te)]:
                for op, cutoff in [('default', .5), ('oof_recall', threshold)]:
                    rows.append(dict(mode=mode, seed=seed, dataset=records[0].get('dataset', 'internal'), split=split, operating_point=op, **metrics(y[ix], p[ix], cutoff)))
                predictions.extend(dict(mode=mode, seed=seed, split=split, record_id=records[i]['record_id'], insoluble=int(y[i]), score=float(p[i])) for i in ix)
            for rr, xx in external:
                pp = estimator.predict(xx)
                valid = np.array([r['insoluble'] is not None for r in rr]); yy = np.array([r['insoluble'] or 0 for r in rr])
                for op, cutoff in [('default', .5), ('oof_recall', threshold)]:
                    rows.append(dict(mode=mode, seed=seed, dataset=rr[0]['dataset'], split='external', operating_point=op, **metrics(yy[valid], pp[valid], cutoff)))
                quantitative = [i for i,r in enumerate(rr) if r.get('solubility_fraction') is not None]
                if len(quantitative) > 2:
                    rho = float(spearmanr([-rr[i]['solubility_fraction'] for i in quantitative], pp[quantitative]).statistic)
                    rows.append(dict(mode=mode, seed=seed, dataset=rr[0]['dataset'], split='external_continuous', operating_point='ranking', n=len(quantitative), spearman=rho if np.isfinite(rho) else None))
                predictions.extend(dict(mode=mode, seed=seed, split='external', record_id=r['record_id'], insoluble=r['insoluble'], score=float(v)) for r,v in zip(rr,pp))
            print(f'{mode} seed={seed}: OOF AP={metrics(y[tr],p[tr])["average_precision"]:.3f}; holdout AP={metrics(y[te],p[te])["average_precision"]:.3f}', flush=True)
            write_json(out/f'metrics_{date}.json', rows)
            write_json(out/f'predictions_{date}.json', predictions)
            write_json(out/f'fits_{date}.json', fits)
    write_json(out/f'splits_{date}.json', {'record_ids':[r['record_id'] for r in records], **manifest})
    from .reporting import report
    report(out, date, export_xlsx=not args.json_only)


def fit_final(args):
    """One deployment model; threshold from full-training grouped OOF predictions."""
    records, store, config = setup(args)
    out = Path(args.output); out.mkdir(parents=True, exist_ok=False)
    data = store.align(records, config['architecture']=='bilstm', config['profile']=='structure')
    y = np.array([r['insoluble'] for r in records], np.float32); groups = np.array([r['group_id'] for r in records])
    from sklearn.model_selection import StratifiedGroupKFold
    p = np.full(len(y), np.nan); fits = []
    for fold, (tr, va) in enumerate(StratifiedGroupKFold(3, shuffle=True, random_state=args.seed+1000).split(y,y,groups)):
        est = Estimator(config, store, args.seed+fold*10000, args.device).fit(subset(data,tr), y[tr], groups[tr])
        p[va] = est.predict(subset(data,va)); fits.append(est.metadata)
    threshold = choose_threshold(y,p,args.recall)
    est = Estimator(config, store, args.seed, args.device).fit(data,y,groups)
    est.save(out/f'deployment_{args.date}.pt', threshold)
    write_json(out/f'deployment_fit_{args.date}.json', dict(oof_metrics=metrics(y,p,threshold), threshold_selection='full training grouped OOF, apparent OOF threshold metrics', fits=fits, final_fit=est.metadata, independent_holdout_metrics='Run evaluate separately; no holdout created by fit-final'))
    prediction_rows=[dict(record_id=r['record_id'],insoluble=int(label),oof_score=float(score)) for r,label,score in zip(records,y,p)]
    write_json(out/f'deployment_oof_{args.date}.json',prediction_rows)
    if not args.json_only:
        from .reporting import export_tables
        export_tables(out/f'deployment_oof_{args.date}.xlsx', {'OOF_predictions':prediction_rows,'OOF_metrics':[metrics(y,p,threshold)]})


def predict(args):
    records = read_records(args.records)
    store = FeatureStore(args.features, args.structure)
    estimator = Estimator.load(args.checkpoint, store, args.device)
    x = store.align(records, estimator.need_tokens, estimator.config['profile']=='structure')
    p = estimator.predict(x)
    out = Path(args.output); out.mkdir(parents=True, exist_ok=True)
    rows = [dict(record_id=r['record_id'], sequence=r['sequence'], insolubility_score=float(v), threshold=estimator.threshold, reject_before_synthesis=bool(v>=estimator.threshold)) for r,v in zip(records,p)]
    write_json(out/f'predictions_{args.date}.json',rows)
    if not args.json_only:
        from .reporting import export_tables
        export_tables(out/f'predictions_{args.date}.xlsx', {'Predictions':rows})


def main():
    parser = argparse.ArgumentParser(description=__doc__); sub = parser.add_subparsers(dest='command',required=True)
    for name in ['evaluate','fit-final','predict']:
        p = sub.add_parser(name); p.add_argument('--records',required=True);p.add_argument('--features',required=True)
        p.add_argument('--structure'); p.add_argument('--output',required=True);p.add_argument('--device',default='cpu',choices=['cpu','cuda','mps'])
        p.add_argument('--date',default=datetime.now().strftime('%Y%m%d'));p.add_argument('--json-only',action='store_true')
        if name != 'predict':
            p.add_argument('--config',required=True);p.add_argument('--recall',type=float,default=.9)
        else:
            p.add_argument('--checkpoint',required=True)
    p = sub.choices['evaluate'];p.add_argument('--splits');p.add_argument('--modes',nargs='+',choices=['random','grouped'],default=['random','grouped']);p.add_argument('--external',nargs='*',default=[]);p.add_argument('--max-epochs',type=int)
    sub.choices['fit-final'].add_argument('--seed',type=int,default=11)
    args=parser.parse_args()
    {'evaluate':train,'fit-final':fit_final,'predict':predict}[args.command](args)


if __name__ == '__main__':
    main()
```

## src/solubility_public_data/data.py

```python
"""ID/sequence-checked data interfaces. Source metadata never enters features."""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import torch

AA = set('ACDEFGHIKLMNPQRSTVWY')
SEEDS = [11, 29, 47, 71, 101]


def read_records(path, training=False):
    path = Path(path)
    if path.suffix == '.xlsx':
        records = pd.read_excel(path, sheet_name='Records', keep_default_na=False).to_dict('records')
    else:
        records = json.loads(path.read_text())
    if not records:
        raise ValueError('Empty records')
    ids = [str(r['record_id']) for r in records]
    if len(set(ids)) != len(ids):
        raise ValueError('Duplicate record IDs')
    for r in records:
        seq = r['sequence']
        if not isinstance(seq, str) or not 20 <= len(seq) <= 100 or set(seq) - AA:
            raise ValueError('Require canonical 20-100 aa sequences, without silent truncation')
        label = r.get('insoluble')
        if label is None or label == '' or pd.isna(label):
            r['insoluble'] = None
        elif label not in (0, 1):
            raise ValueError('insoluble must be 1, 0, or missing')
        if 'solubility_fraction' in r and (r['solubility_fraction'] == '' or r['solubility_fraction'] is None or pd.isna(r['solubility_fraction'])):
            r['solubility_fraction'] = None
        if training:
            if r.get('role') != 'train' or r['insoluble'] is None:
                raise ValueError('Training requires role=train and binary labels; external records prohibited')
            if r.get('group_id') is None or r['group_id'] == '' or pd.isna(r['group_id']):
                raise ValueError('Training requires sequence/scaffold family group_id')
    if training and (len({r['sequence'] for r in records}) != len(records) or {r['insoluble'] for r in records} != {0, 1}):
        raise ValueError('Training requires unique sequences and both classes')
    return records


def subset(data, indices):
    return {k: v[indices] for k, v in data.items()}


class FeatureStore:
    """Portable pooled NPZ and optional memory-mapped residue array."""
    def __init__(self, directory, structure=None):
        self.directory = Path(directory)
        with np.load(self.directory / 'pooled_20261108.npz', allow_pickle=False) as a:
            self.pooled = {k: a[k] for k in a.files}
        ids = self.pooled['ids'].astype(str)
        if len(set(ids)) != len(ids):
            raise ValueError('Duplicate feature IDs')
        self.lookup = {s: i for i, s in enumerate(ids)}
        n = len(ids)
        for key, width in [('desc', 202), ('esm650', 1280), ('esm650_max', 1280)]:
            if self.pooled[key].shape != (n, width) or not np.isfinite(self.pooled[key]).all():
                raise ValueError(f'Invalid {key} feature matrix')
        self.tokens = None
        if (self.directory / 'residues_20261108.npy').exists():
            self.tokens = np.load(self.directory / 'residues_20261108.npy', mmap_mode='r', allow_pickle=False)
            off = self.pooled['offsets']
            if off.shape != (n+1,) or off[0] != 0 or off[-1] != len(self.tokens) or not np.array_equal(np.diff(off), [len(s) for s in self.pooled['sequences']]):
                raise ValueError('Residue offsets/sequence lengths mismatch')
            if self.tokens.shape[1:] != (1280,):
                raise ValueError('ESM650 token width must be 1280')
        self.structure = None
        if structure:
            with np.load(structure, allow_pickle=False) as s:
                self.structure = {k: s[k] for k in s.files}
            self._validate_structure()

    def _validate_structure(self):
        s = self.structure
        ids = s['ids'].astype(str); n = len(ids)
        if len(set(ids)) != n:
            raise ValueError('Duplicate structure IDs')
        self.structure_lookup = {v: i for i, v in enumerate(ids)}
        if s['global_values'].ndim != 2 or s['global_values'].shape[0] != n or s['quality'].shape != (n, 3):
            raise ValueError('Bad structure global/quality shape')
        self.structure_dims = {'global': s['global_values'].shape[1]}
        for key in ['global', 'residue', 'patch']:
            a = s[key+'_available']
            if a.shape != (n,) or not np.isin(a, [0, 1]).all():
                raise ValueError('Availability flags must be binary per protein')
            if key == 'global':
                continue
            v, o, mask = s[key+'_values'], s[key+'_offsets'], s[key+'_mask']
            if v.ndim != 2 or o.shape != (n+1,) or o[0] != 0 or o[-1] != len(v) or np.any(np.diff(o) < 0) or mask.shape != (len(v),) or not np.isin(mask, [0, 1]).all():
                raise ValueError('Invalid ragged structure schema')
            self.structure_dims[key] = v.shape[1]
        if not np.isfinite(s['quality']).all() or np.any((s['quality'] < 0) | (s['quality'] > 1)):
            raise ValueError('Declare quality values in [0,1], never use assay outcomes')
        for key in self.structure_dims:
            names = s[key+'_feature_names']
            if len(names) != self.structure_dims[key] or len(set(names)) != len(names):
                raise ValueError('Feature names must be unique and match dimensions')
        self.structure_schema = {k: s[k+'_feature_names'].tolist() for k in self.structure_dims}

    def align(self, records, need_tokens=False, need_structure=False):
        ix = np.asarray([self.lookup[str(r['record_id'])] for r in records])
        sequences = np.asarray([r['sequence'] for r in records])
        if not np.array_equal(sequences, self.pooled['sequences'][ix]):
            raise ValueError('Sequence mismatch for feature ID')
        if need_tokens and self.tokens is None:
            raise ValueError('BiLSTM requires residue cache; download or extract features first')
        data = {k: self.pooled[k][ix] for k in ['desc', 'esm650', 'esm650_max']}
        data['rows'] = ix
        if need_structure:
            if self.structure is None:
                raise ValueError('Structure profile requires a structure NPZ sidecar')
            si = np.asarray([self.structure_lookup[str(r['record_id'])] for r in records])
            sha = [hashlib.sha256(s.encode()).hexdigest() for s in sequences]
            if not np.array_equal(self.structure['sequence_sha256'][si], sha):
                raise ValueError('Structure sequence hash mismatch')
            offsets = self.structure['residue_offsets']
            # Available residue structures must align one item to every sequence residue.
            for i, seq in zip(si, sequences):
                if self.structure['residue_available'][i] and offsets[i+1]-offsets[i] != len(seq):
                    raise ValueError('Structure residue coordinates must align to sequence')
            data['structure_rows'] = si
        return data

    def scalers(self, data, ix):
        result = {}
        for key in ['desc', 'esm650', 'esm650_max']:
            x = data[key][ix].astype(np.float64)
            result[key] = self._stats(x)
        if 'structure_rows' in data:
            s, rows = self.structure, data['structure_rows'][ix]
            for key in ['global', 'residue', 'patch']:
                pieces = []
                for row in rows:
                    if not s[key+'_available'][row]:
                        continue
                    if key == 'global':
                        pieces.append(s['global_values'][row:row+1])
                    else:
                        lo, hi = s[key+'_offsets'][row:row+2]
                        pieces.append(s[key+'_values'][lo:hi][s[key+'_mask'][lo:hi].astype(bool)])
                width = self.structure_dims[key]
                x = np.concatenate(pieces) if pieces else np.empty((0, width))
                result['structure_'+key] = self._stats(x)
        return result

    @staticmethod
    def _stats(x):
        # Missing individual structure values: fit-only mean imputation, then z-score.
        valid = np.isfinite(x)
        count = valid.sum(0)
        mean = np.where(valid, x, 0.).sum(0) / np.maximum(count, 1)
        var = np.where(valid, x-mean, 0.)**2
        scale = np.sqrt(var.sum(0) / np.maximum(count, 1))
        scale[scale < 1e-12] = 1.
        return {'mean': mean, 'scale': scale}

    def batch(self, data, ix, scalers, device='cpu', need_tokens=False):
        def tensor(a, dtype=torch.float32):
            return torch.as_tensor(np.ascontiguousarray(a), dtype=dtype, device=device)
        def norm(a, key):
            stat = scalers[key]
            return (np.where(np.isfinite(a), a, stat['mean'])-stat['mean'])/stat['scale']
        b = {k: tensor(norm(data[k][ix], k)) for k in ['desc', 'esm650', 'esm650_max']}
        if need_tokens:
            rows = data['rows'][ix]; offsets = self.pooled['offsets']
            lengths = offsets[rows+1]-offsets[rows]
            values = np.zeros((len(ix), int(lengths.max()), 1280), np.float32)
            for j, row in enumerate(rows):
                raw = self.tokens[offsets[row]:offsets[row+1]]
                if not np.isfinite(raw).all():
                    raise ValueError('Nonfinite valid residue states')
                values[j, :lengths[j]] = raw
            b.update(residue=tensor(values), mask=tensor(np.arange(values.shape[1])[None, :] < lengths[:, None], torch.bool), lengths=torch.as_tensor(lengths, dtype=torch.long))
        if 'structure_rows' in data:
            s, rows = self.structure, data['structure_rows'][ix]
            available = np.column_stack([s[k+'_available'][rows] for k in ['global', 'residue', 'patch']])
            b['structure_available'] = tensor(available)
            b['structure_quality'] = tensor(s['quality'][rows] * available)
            b['structure_global'] = tensor(norm(s['global_values'][rows], 'structure_global') * available[:, :1])
            for key in ['residue', 'patch']:
                offsets = s[key+'_offsets']; lengths = offsets[rows+1]-offsets[rows]
                width = max(1, int(lengths.max()))
                values = np.zeros((len(rows), width, self.structure_dims[key]), np.float32)
                mask = np.zeros((len(rows), width), bool)
                for j, row in enumerate(rows):
                    lo, hi = offsets[row:row+2]
                    values[j, :hi-lo] = norm(s[key+'_values'][lo:hi], 'structure_'+key)
                    mask[j, :hi-lo] = s[key+'_mask'][lo:hi].astype(bool) & bool(s[key+'_available'][row])
                b['structure_'+key] = tensor(values)
                b['structure_'+key+'_mask'] = tensor(mask, torch.bool)
        return b
```

## src/solubility_public_data/descriptors.py

```python
"""Sequence-only solubility descriptors and train-only neural classifiers.

Created 2026-11-06. All features use the input sequence only. Experimental solubility, expression
yield, assay readouts, and source identities must never be passed as X.
The classifier chooses its epoch count on an internal training-only validation
split, then refits from scratch on all rows supplied to fit(). Outer validation
and holdout rows must be supplied only to predict_proba().
"""

from __future__ import annotations

import copy
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.SeqUtils import ProtParamData

AA = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_ID = {aa: i + 1 for i, aa in enumerate(AA)}
HYDROPHOBIC = set("AILMFWVY")
AA_GROUPS = {
    "hydrophobic": HYDROPHOBIC,
    "aromatic": set("FWY"),
    "positive": set("KRH"),
    "negative": set("DE"),
    "small": set("AGST"),
    "polar_uncharged": set("NQST"),
    "pro_gly": set("PG"),
}


def _longest_run(sequence, residue_set):
    best = current = 0
    for residue in sequence:
        current = current + 1 if residue in residue_set else 0
        best = max(best, current)
    return best


def sequence_features(sequences, include_dipeptides=False):
    """Return interpretable descriptors in fixed, deterministic column order.

    Janin and Kyte-Doolittle scales are the BioPython ProtParamData tables.
    The pH 7 charge is an explicit approximate Henderson-Hasselbalch descriptor,
    not an observed charge or a pI estimate. Terminal windows characterize
    local sequence composition without using assay or source metadata.
    """
    rows = []
    for sequence in sequences:
        sequence = str(sequence).strip().upper()
        if not sequence or set(sequence).difference(AA):
            raise ValueError(f"Expected nonempty canonical sequence, got {sequence!r}")
        n = len(sequence)
        counts = Counter(sequence)
        kd = np.fromiter((ProtParamData.kd[a] for a in sequence), float)
        janin = np.fromiter((ProtParamData.ja[a] for a in sequence), float)
        fracs = np.array([counts[a] / n for a in AA])
        record = {
            "length": n,
            "janin_mean": float(janin.mean()),
            "kd_mean": float(kd.mean()),
            "kd_std": float(kd.std()),
            "composition_entropy": float(-np.sum(fracs[fracs > 0] * np.log(fracs[fracs > 0]))),
            "unique_residues": len(counts),
            "approx_charge_pH7": (
                1 / (1 + 10 ** (7 - 9.69)) - 1 / (1 + 10 ** (2.34 - 7))
                + counts["K"] / (1 + 10 ** (7 - 10.5))
                + counts["R"] / (1 + 10 ** (7 - 12.4))
                + counts["H"] / (1 + 10 ** (7 - 6.0))
                - counts["D"] / (1 + 10 ** (3.86 - 7))
                - counts["E"] / (1 + 10 ** (4.25 - 7))
                - counts["C"] / (1 + 10 ** (8.33 - 7))
                - counts["Y"] / (1 + 10 ** (10.07 - 7))
            ),
        }
        for aa in AA:
            record[f"count_{aa}"] = counts[aa]
        for name, subsequence in [
            ("global", sequence), ("n5", sequence[:5]),
            ("c5", sequence[-5:]), ("c10", sequence[-10:]),
            ("c20", sequence[-20:]),
        ]:
            subcount = Counter(subsequence)
            for aa in AA:
                record[f"{name}_frac_{aa}"] = subcount[aa] / len(subsequence)
            record[f"{name}_kd_mean"] = np.mean([ProtParamData.kd[a] for a in subsequence])
        for aa in AA:
            record[f"nterm_{aa}"] = int(sequence[0] == aa)
            record[f"cterm_{aa}"] = int(sequence[-1] == aa)
        for name, group in AA_GROUPS.items():
            record[f"{name}_fraction"] = sum(counts[a] for a in group) / n
            record[f"{name}_longest_run"] = _longest_run(sequence, group)
        for width in [3, 5, 7, 10]:
            size = min(width, n)
            rolling = np.convolve(kd, np.ones(size) / size, mode="valid")
            record[f"kd_window{width}_max"] = rolling.max()
            record[f"kd_window{width}_min"] = rolling.min()
        for motif in ["DP", "DG", "DS", "DT", "NG", "NS", "PP", "CC"]:
            record[f"motif_{motif}_fraction"] = sum(
                sequence[i:i + 2] == motif for i in range(n - 1)
            ) / max(n - 1, 1)
        if include_dipeptides:
            pairs = Counter(sequence[i:i + 2] for i in range(n - 1))
            for a in AA:
                for b in AA:
                    record[f"dipeptide_{a}{b}"] = pairs[a + b] / max(n - 1, 1)
        rows.append(record)
    return pd.DataFrame.from_records(rows).astype(np.float32)
```

## src/solubility_public_data/evaluation.py

```python
"""Five distinct holdouts and training-only OOF threshold selection."""
import hashlib
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (average_precision_score, roc_auc_score, confusion_matrix,
                            matthews_corrcoef, brier_score_loss, log_loss)
from .data import SEEDS


def make_splits(records, mode):
    y = np.array([r['insoluble'] for r in records]); groups = np.array([r['group_id'] for r in records])
    result = {}
    for seed in SEEDS:
        if mode == 'random':
            tr, te = train_test_split(np.arange(len(y)), test_size=.2, stratify=y, random_state=seed)
            cv = StratifiedKFold(3, shuffle=True, random_state=seed+1000)
            folds_iter = cv.split(tr, y[tr])
        elif mode == 'grouped':
            tr, te = next(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(y, y, groups))
            folds_iter = StratifiedGroupKFold(3, shuffle=True, random_state=seed+1000).split(tr, y[tr], groups[tr])
        else:
            raise ValueError(mode)
        folds = np.full(len(y), -1, int)
        for k, (_, valid) in enumerate(folds_iter):
            folds[tr[valid]] = k
        result[str(seed)] = {'train': tr.tolist(), 'test': te.tolist(), 'folds': folds.tolist()}
    validate_splits(records, result, mode)
    return result


def validate_splits(records, splits, mode):
    n = len(records); y = np.array([r['insoluble'] for r in records]); g = np.array([r['group_id'] for r in records]); hashes = []
    if set(splits) != set(map(str, SEEDS)):
        raise ValueError('Exactly five prescribed seeds required')
    for s in splits.values():
        tr, te, f = np.array(s['train']), np.array(s['test']), np.array(s['folds'])
        if f.shape != (n,) or len(set(tr)) != len(tr) or len(set(te)) != len(te) or set(tr) & set(te) or set(tr) | set(te) != set(range(n)):
            raise ValueError('Invalid split partition')
        if set(f[tr]) != {0, 1, 2} or not np.all(f[te] == -1):
            raise ValueError('Invalid OOF assignments')
        for a, b in [(tr, te)]+[(tr[f[tr] != k], tr[f[tr] == k]) for k in range(3)]:
            if len(np.unique(y[a])) != 2 or len(np.unique(y[b])) != 2:
                raise ValueError('Every fitting/evaluation partition needs both classes')
            if mode == 'grouped' and set(g[a]) & set(g[b]):
                raise ValueError('Family leakage')
        hashes.append(hashlib.sha256(np.sort(te).astype('<i8').tobytes()).hexdigest())
    if len(set(hashes)) != 5:
        raise ValueError('Holdout memberships must differ across all five seeds')


def choose_threshold(y, p, recall=.9):
    """Largest score cutoff attaining requested insoluble recall on training OOF."""
    positive = np.sort(np.asarray(p)[np.asarray(y) == 1])
    if not len(positive) or not 0 < recall <= 1:
        raise ValueError('Positive labels and valid recall required')
    return float(positive[max(0, len(positive)-int(np.ceil(recall*len(positive))))])


def metrics(y, p, threshold=.5):
    y, p = np.asarray(y, int), np.asarray(p, float)
    if not len(y) or not np.isfinite(p).all():
        raise ValueError('Empty/nonfinite predictions')
    pred = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    divide = lambda a, b: float(a/b) if b else None
    both = len(np.unique(y)) == 2
    return dict(n=len(y), n_insoluble=int(y.sum()), prevalence=float(y.mean()),
        auroc=float(roc_auc_score(y, p)) if both else None,
        average_precision=float(average_precision_score(y, p)) if y.sum() else None,
        brier=float(brier_score_loss(y, p)), log_loss=float(log_loss(y, p, labels=[0, 1])),
        threshold=float(threshold), precision=divide(tp, tp+fp), recall=divide(tp, tp+fn),
        specificity=divide(tn, tn+fp), mcc=float(matthews_corrcoef(y, pred)),
        true_positive=int(tp), false_positive=int(fp), true_negative=int(tn), false_negative=int(fn),
        rejection_fraction=float(pred.mean()), soluble_retention=divide(tn, tn+fp),
        accepted_insoluble_fraction=divide(fn, tn+fn), accepted_n=int(tn+fn))
```

## src/solubility_public_data/models.py

```python
"""Two gated residual architectures; public and optional structure profiles.

The public profiles reproduce the previously studied head topology. Compact
structure profiles are proposals, not experimentally validated improvements.
"""
import torch
from torch import nn


def block(n, m, p=.2):
    return nn.Sequential(nn.Linear(n, m), nn.LayerNorm(m), nn.GELU(), nn.Dropout(p))


def masked_pool(x, mask):
    """Mean/max over valid positions; empty sets map to exact zeros."""
    mask = mask.bool().unsqueeze(-1)
    clean = x.masked_fill(~mask, 0.)
    mean = clean.sum(1) / mask.sum(1).clamp_min(1)
    maximum = x.masked_fill(~mask, -torch.inf).max(1).values
    maximum = torch.where(mask.any(1), maximum, torch.zeros_like(maximum))
    return torch.cat([mean, maximum], -1)


class SetEncoder(nn.Module):
    """Shared item MLP, permutation-invariant masked pooling, 16-D output."""
    def __init__(self, width):
        super().__init__()
        self.items = block(width, 32)
        self.output = block(64, 16)

    def forward(self, x, mask):
        h = self.items(x.masked_fill(~mask.unsqueeze(-1), 0.))
        result = self.output(masked_pool(h, mask))
        return result * mask.any(1, keepdim=True)


class StructureEncoder(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.global_branch = block(dims['global'], 32)
        self.residue_branch = SetEncoder(dims['residue'])
        self.patch_branch = SetEncoder(dims['patch'])
        self.norm = nn.LayerNorm(64)

    def forward(self, b):
        global_h = self.global_branch(b['structure_global']) * b['structure_available'][:, :1]
        residue_h = self.residue_branch(b['structure_residue'], b['structure_residue_mask'])
        patch_h = self.patch_branch(b['structure_patch'], b['structure_patch_mask'])
        return self.norm(torch.cat([global_h, residue_h, patch_h], 1))


class ResidueBiLSTM(nn.Module):
    """One bidirectional layer, 32 units/direction, no padding in recurrence."""
    def __init__(self):
        super().__init__()
        self.projection = block(1280, 64, .1)
        self.encoder = nn.LSTM(64, 32, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(.1)
        self.output = block(128, 64)

    def forward(self, b):
        mask = b['mask']
        h = self.projection(b['residue'].masked_fill(~mask.unsqueeze(-1), 0.))
        packed = nn.utils.rnn.pack_padded_sequence(h, b['lengths'].cpu(), batch_first=True, enforce_sorted=False)
        encoded, _ = self.encoder(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(encoded, batch_first=True, total_length=h.shape[1])
        return self.output(masked_pool(self.dropout(h), mask))


class FusionModel(nn.Module):
    def __init__(self, architecture='residual_mlp', profile='public', structure_dims=None):
        super().__init__()
        if architecture not in {'residual_mlp', 'bilstm'} or profile not in {'public', 'structure'}:
            raise ValueError('Unknown architecture/profile')
        self.architecture, self.profile = architecture, profile
        self.structure_dims = structure_dims
        self.esm = block(2560, 64)
        if profile == 'public':
            self.other = block(202, 64)
        else:
            if not structure_dims or any(structure_dims[k] < 1 for k in ['global', 'residue', 'patch']):
                raise ValueError('Specify positive global/residue/patch dimensions')
            self.other = StructureEncoder(structure_dims)
        self.sequence = ResidueBiLSTM() if architecture == 'bilstm' else None
        views = 3 if self.sequence else 2
        width = 64 * views
        compact = profile == 'structure'
        quality_dim = 6 if compact else 0  # availability 3 + declared quality 3
        self.gate = nn.Sequential(nn.Linear(width + quality_dim, 16 if compact else 32), nn.GELU(), nn.Linear(16 if compact else 32, views))
        hidden = 64 if compact else 96
        count = 1 if compact else 2
        self.residuals = nn.ModuleList([nn.Sequential(nn.LayerNorm(width), nn.Linear(width, hidden), nn.GELU(), nn.Dropout(.2), nn.Linear(hidden, width)) for _ in range(count)])
        self.residual_gates = nn.Parameter(torch.zeros(count, width))
        self.output = block(width, 32 if compact else 96)
        self.head = nn.Linear(32 if compact else 96, 1)

    def sample_keep(self, b):
        n = len(b['esm650']); v = 3 if self.sequence else 2
        keep = torch.ones(n, v, dtype=torch.bool, device=b['esm650'].device)
        if self.profile == 'structure':
            keep[:, 0] = b['structure_available'].bool().any(1)
            if self.training:
                keep[:, 0] &= torch.rand(n, device=keep.device) >= .15
        elif self.training:
            keep = torch.rand(n, v, device=keep.device) >= .15
            keep[:, 1] |= ~keep.any(1)
        return keep

    def forward(self, b, keep=None):
        # Ordering is descriptor/structure, pooled ESM, optional ESM BiLSTM.
        h = self.other(b['desc']) if self.profile == 'public' else self.other(b)
        views = [h, self.esm(torch.cat([b['esm650'], b['esm650_max']], 1))]
        if self.sequence:
            views.append(self.sequence(b))
        views = torch.stack(views, 1)
        keep = self.sample_keep(b) if keep is None else keep
        visible = views * keep.unsqueeze(-1)
        gate_inputs = visible.flatten(1)
        if self.profile == 'structure':
            gate_inputs = torch.cat([gate_inputs, b['structure_available'], b['structure_quality']], 1)
        gates = torch.softmax(self.gate(gate_inputs).masked_fill(~keep, -torch.inf), 1)
        x = (visible * gates.unsqueeze(-1)).flatten(1)
        for i, layer in enumerate(self.residuals):
            x = x + torch.sigmoid(self.residual_gates[i]) * layer(x)
        return {'logit': self.head(self.output(x)).squeeze(-1), 'gates': gates}


def symmetric_kl(a, b):
    la = torch.stack([nn.functional.logsigmoid(-a), nn.functional.logsigmoid(a)], -1)
    lb = torch.stack([nn.functional.logsigmoid(-b), nn.functional.logsigmoid(b)], -1)
    return .5 * ((la.exp() * (la - lb)).sum(-1) + (lb.exp() * (lb - la)).sum(-1)).mean()
```

## src/solubility_public_data/reporting.py

```python
"""Portable training reports; PNG figures, XLSX tables and Markdown narrative."""
import json
from pathlib import Path
import pandas as pd


def export_tables(path, tables):
    # Runtime export is intentionally portable; repository workbooks were authored
    # and independently verified using the artifact-tool release builder.
    with pd.ExcelWriter(path, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_formulas': False, 'strings_to_urls': False}}) as writer:
        for name, rows in tables.items():
            frame = pd.DataFrame(rows); frame.to_excel(writer, sheet_name=name[:31],index=False)
            sheet=writer.sheets[name[:31]];sheet.freeze_panes(1,0)


def report(directory, date, export_xlsx=True):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    out=Path(directory); frame=pd.DataFrame(json.loads((out/f'metrics_{date}.json').read_text()))
    group=['mode','dataset','split','operating_point']
    selected=['average_precision','auroc','mcc','recall','precision','soluble_retention','accepted_insoluble_fraction','prevalence']
    if 'spearman' in frame:
        selected.append('spearman')
    summary=frame.groupby(group)[selected].agg(['mean','std','count']).reset_index()
    summary.columns=['_'.join(x).rstrip('_') if isinstance(x,tuple) else x for x in summary.columns]
    (out/f'summary_{date}.json').write_text(summary.to_json(orient='records',indent=2)+'\n')
    if export_xlsx:
        export_tables(out/f'results_{date}.xlsx', {'Metrics':frame.to_dict('records'),'Summary':summary.to_dict('records'),'Predictions':json.loads((out/f'predictions_{date}.json').read_text())})
    plot=summary[(summary.operating_point=='default') & summary.split.isin(['oof','holdout','external'])]
    fig,ax=plt.subplots(figsize=(9,max(4,len(plot)*.45+1.5)))
    ax.barh(range(len(plot)),plot.average_precision_mean,xerr=plot.average_precision_std.fillna(0),capsize=3,label='AP mean ± SD')
    ax.scatter(plot.prevalence_mean,range(len(plot)),color='black',marker='|',s=130,label='Positive prevalence',zorder=3)
    ax.set_yticks(range(len(plot)),[f'{r.mode} / {r.dataset} / {r.split}' for r in plot.itertuples()])
    ax.invert_yaxis();ax.legend(loc='lower right',fontsize=8)
    protocol=json.loads((out/f'protocol_{date}.json').read_text())
    title='Five distinct training/holdout partitions; mean ± SD'
    if protocol['epochs_capped_for_smoke']:
        title='SMOKE CHECK ONLY — epoch-capped fits; mean ± SD'
    ax.set_xlim(0,1);ax.set_xlabel('Insoluble average precision');ax.set_title(title,fontsize=11)
    fig.tight_layout();fig.savefig(out/f'performance_{date}.png',dpi=150);plt.close(fig)
    note='SMOKE VALIDATION ONLY: epoch-capped results are not scientific performance estimates.' if protocol['epochs_capped_for_smoke'] else 'Prespecified configuration; no selection on external labels.'
    (out/f'report_{date}.md').write_text(f'# Training report — {date}\n\n{note}\n\nTraining rows: {protocol["n"]}; insoluble: {protocol["n_insoluble"]}; groups: {protocol["n_groups"]}. Seeds: {protocol["seeds"]}.\n\nSee protocol JSON for features/architecture and fits JSON for parameter counts, epoch selection and split membership. OOF predictions come from three training-only folds; five holdouts differ. Random splits allow homologous groups to cross boundaries; grouped splits do not. Repeated holdout SD is descriptive, not an independent-replicate confidence interval. External panels stay fixed, evaluated by five separately trained models.\n\nThresholds use training OOF scores. OOF threshold metrics are apparent after threshold selection; holdout metrics are the assessment. Sigmoid scores are uncalibrated. A target OOF recall does not guarantee future insolubility rejection or an accepted-set contamination rate.\n\n![Performance](performance_{date}.png)\n')
```

## src/solubility_public_data/training.py

```python
"""Group-disjoint inner stopping followed by a freshly initialized full refit."""
import random
import numpy as np
import torch
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from .models import FusionModel, symmetric_kl

DEFAULTS = dict(max_epochs=60, patience=8, batch_size=128, lr=.001,
                weight_decay=.01, gradient_clip=5., rdrop_weight=.5,
                min_delta=1e-5, validation_candidates=100, num_threads=2)


def seed_all(seed, threads=2):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.set_num_threads(threads)
    torch.use_deterministic_algorithms(True)


def inner_split(y, groups, seed, candidates=100):
    choices = []
    for tr, va in GroupShuffleSplit(n_splits=candidates, test_size=.2, random_state=seed+101).split(y, y, groups):
        if len(np.unique(y[tr])) == len(np.unique(y[va])) == 2:
            distance = abs(y[va].mean()-y.mean()) + abs(len(va)/len(y)-.2)
            choices.append((distance, tr, va))
    if not choices:
        raise ValueError('Too few class-supported groups for inner validation')
    _, tr, va = min(choices, key=lambda x: x[0])
    return tr, va


class Estimator:
    def __init__(self, config, store, seed=11, device='cpu'):
        self.config = {**DEFAULTS, **config}
        self.store, self.seed, self.device = store, int(seed), device
        self.need_tokens = self.config['architecture'] == 'bilstm'
        if self.config['profile'] == 'structure':
            self.config['structure_dims'] = store.structure_dims

    def network(self):
        return FusionModel(self.config['architecture'], self.config['profile'], self.config.get('structure_dims')).to(self.device)

    def _predict(self, model, data, ix, scalers):
        model.eval(); chunks = []
        with torch.inference_mode():
            for start in range(0, len(ix), 64):
                batch = self.store.batch(data, ix[start:start+64], scalers, self.device, self.need_tokens)
                chunks.append(torch.sigmoid(model(batch)['logit']).cpu().numpy())
        return np.concatenate(chunks).astype(float) if chunks else np.array([])

    def _run(self, data, y, ix, scalers, epochs, validation=None):
        c = self.config; seed_all(self.seed, c['num_threads'])
        model = self.network()
        optimizer = torch.optim.AdamW(model.parameters(), lr=c['lr'], weight_decay=c['weight_decay'])
        w = float(np.sqrt((len(ix)-y[ix].sum())/y[ix].sum()))
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w, device=self.device))
        best, best_epoch, stale, history = -1., 1, 0, []
        for epoch in range(1, int(epochs)+1):
            model.train(); order = ix[torch.randperm(len(ix)).numpy()]; total = 0.
            for start in range(0, len(order), c['batch_size']):
                rows = order[start:start+c['batch_size']]
                batch = self.store.batch(data, rows, scalers, self.device, self.need_tokens)
                labels = torch.as_tensor(y[rows], device=self.device)
                optimizer.zero_grad(set_to_none=True)
                # Compact structure profile shares its modality mask across R-Drop passes.
                # Public profiles preserve independently redrawn historical view masks.
                keep = model.sample_keep(batch) if c['profile'] == 'structure' else None
                a = model(batch, keep)['logit']; loss = loss_fn(a, labels)
                if c['rdrop_weight']:
                    b = model(batch, keep)['logit']
                    loss = .5*(loss+loss_fn(b, labels)) + c['rdrop_weight']*symmetric_kl(a, b)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite loss')
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), c['gradient_clip']); optimizer.step()
                total += float(loss.detach().cpu())*len(rows)
            row = {'epoch': epoch, 'training_loss': total/len(ix)}
            if validation is not None:
                ap = float(average_precision_score(y[validation], self._predict(model, data, validation, scalers)))
                row['inner_insoluble_ap'] = ap
                if ap > best+c['min_delta']:
                    best, best_epoch, stale = ap, epoch, 0
                else:
                    stale += 1
            history.append(row)
            if validation is not None and stale >= c['patience']:
                break
        return model, best_epoch if validation is not None else int(epochs), history

    def fit(self, data, y, groups):
        y = np.asarray(y, np.float32); groups = np.asarray(groups).astype(str)
        if y.shape != groups.shape or set(np.unique(y)) != {0., 1.}:
            raise ValueError('Aligned groups and both classes required')
        tr, va = inner_split(y, groups, self.seed, self.config['validation_candidates'])
        inner_scalers = self.store.scalers(data, tr)
        _, epochs, history = self._run(data, y, tr, inner_scalers, self.config['max_epochs'], va)
        self.scalers = self.store.scalers(data, np.arange(len(y)))
        self.model, _, refit = self._run(data, y, np.arange(len(y)), self.scalers, epochs)
        self.model.eval()
        self.metadata = dict(n_training=len(y), n_insoluble=int(y.sum()), n_soluble=int((1-y).sum()),
            n_groups=len(set(groups)), seed=self.seed, selected_epochs=epochs,
            positive_weight=float(np.sqrt((len(y)-y.sum())/y.sum())),
            trainable_parameters=sum(p.numel() for p in self.model.parameters()),
            inner_train=tr.tolist(), inner_validation=va.tolist(),
            inner_group_overlap=len(set(groups[tr]) & set(groups[va])),
            inner_history=history, refit_history=refit, reinitialized_before_refit=True,
            positive_class='insoluble', scores_calibrated=False)
        return self

    def predict(self, data):
        return self._predict(self.model, data, np.arange(len(data['rows'])), self.scalers)

    def save(self, path, threshold=.5):
        torch.save(dict(format='mini_solubility_v1', config=self.config, seed=self.seed,
            state={k: v.detach().cpu() for k, v in self.model.state_dict().items()},
            scalers={k: {a: torch.as_tensor(v) for a, v in s.items()} for k, s in self.scalers.items()},
            metadata=self.metadata, threshold=float(threshold),
            structure_schema=getattr(self.store, 'structure_schema', None)), path)

    @classmethod
    def load(cls, path, store, device='cpu'):
        saved = torch.load(path, map_location='cpu', weights_only=True)
        if saved['format'] != 'mini_solubility_v1':
            raise ValueError('Unknown checkpoint format')
        if saved['structure_schema'] != getattr(store, 'structure_schema', None):
            raise ValueError('Structure feature names/order mismatch')
        obj = cls(saved['config'], store, saved['seed'], device)
        obj.model = obj.network(); obj.model.load_state_dict(saved['state']); obj.model.eval()
        obj.scalers = {k: {a: v.numpy() for a, v in s.items()} for k, s in saved['scalers'].items()}
        obj.metadata, obj.threshold = saved['metadata'], saved['threshold']
        return obj
```

## scripts/build_notebooks.py

```python
"""Generate a runnable quickstart and an annotated, complete source notebook."""
import argparse,hashlib,json
from pathlib import Path
import nbformat as nbf


def build(root):
    md=nbf.v4.new_markdown_cell;code=nbf.v4.new_code_cell
    quick=nbf.v4.new_notebook(cells=[
        md('# Mini-protein insolubility — quickstart\n\nCreated 2026-11-08. Insoluble = 1. Run from the repository or notebooks folder. Install `pip install -e ".[dev,esm]"` first. Full fitting is opt-in; the default cells only inspect public data and model topology.'),
        code("from pathlib import Path\nimport sys, json\nroot = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent\nsys.path.insert(0, str(root / 'src'))\nfrom solubility_public_data.data import read_records\nfrom solubility_public_data.models import FusionModel\nrecords = read_records(root / 'data/public/psi_records_20261108.xlsx', training=True)\nprint({'n': len(records), 'insoluble': sum(r['insoluble'] for r in records), 'groups': len({r['group_id'] for r in records})})"),
        md('## Two public heads\n\nFrozen ESM650 mean/max inputs; BiLSTM also reads frozen residue states. The source notebook documents every module. Structure profiles require measured or predicted structure features supplied separately.'),
        code("for architecture in ['residual_mlp', 'bilstm']:\n    model = FusionModel(architecture)\n    print(architecture, sum(p.numel() for p in model.parameters()))\n    print(model)"),
        md('## Download the feature cache\n\nThis verifies archive and member hashes. It downloads a large public asset only when enabled.'),
        code("import subprocess\nDOWNLOAD_FEATURES = False\nif DOWNLOAD_FEATURES:\n    subprocess.run([sys.executable, 'scripts/download_features.py'], cwd=root, check=True)"),
        md('## Prespecified five-seed evaluation\n\nBoth random and family-grouped splits; three-fold OOF within each training partition. Full fitting may take substantial CPU time. External labels are used only for scoring.'),
        code("RUN_TRAINING = False\nif RUN_TRAINING:\n    for architecture in ['residual_mlp', 'bilstm']:\n        subprocess.run([sys.executable, '-m', 'solubility_public_data.cli', 'evaluate', '--records', 'data/public/psi_records_20261108.xlsx', '--features', 'features/public', '--config', f'configs/{architecture}_public.json', '--splits', 'data/splits/psi_five_seed_splits_20261108.json', '--external', 'data/public/garcia_records_20261108.xlsx', 'data/public/nesg_records_20261108.xlsx', 'data/public/esol_records_20261108.xlsx', '--output', f'runs/{architecture}_20261108'], cwd=root, check=True)"),
        md('## Deployment\n\nUse `solubility fit-final` after evaluating the chosen architecture. This fits one model and sets its cutoff from full-training grouped OOF scores. Use `solubility predict` for new sequences. See `training_instruction.md` for internal structure schema and threshold interpretation.')
    ])
    quick.metadata={'kernelspec':{'display_name':'Python 3','language':'python','name':'python3'},'language_info':{'name':'python'}}
    files=sorted((root/'src').rglob('*.py'))+sorted((root/'scripts').glob('*.py'))+sorted((root/'tests').glob('*.py'))
    cells=[md('# Complete source and training protocol — 2026-11-08\n\nAll Python modules, scripts and tests are merged below in readable functional sections. These are source listings, not execution cells: relative imports and command-line entry points belong to their files. Use the companion quickstart for execution. Rebuild with `python scripts/build_notebooks.py` after changing source files.')]
    hashes={}
    for path in files:
        relative=path.relative_to(root).as_posix();source=path.read_text();hashes[relative]=hashlib.sha256(source.encode()).hexdigest()
        cells.append(md(f'## {relative}\n\n```python\n{source.rstrip()}\n```'))
    source=nbf.v4.new_notebook(cells=cells);source.metadata={'source_sha256':hashes}
    return {'quickstart_20261108.ipynb':quick,'complete_source_20261108.ipynb':source}


def main():
    p=argparse.ArgumentParser();p.add_argument('--check',action='store_true');a=p.parse_args();root=Path(__file__).resolve().parents[1]
    for name,book in build(root).items():
        nbf.validate(book);path=root/'notebooks'/name
        if a.check:
            old=nbf.read(path,as_version=4);nbf.validate(old)
            assert [c.source for c in old.cells]==[c.source for c in book.cells], f'Stale notebook: {name}'
        else:nbf.write(book,path)
    print('Notebook sources validated' if a.check else 'Notebooks generated')


if __name__=='__main__':main()
```

## scripts/download_features.py

```python
"""Download the public release cache with SHA-256 and safe member validation."""
import argparse, hashlib, json, shutil, tarfile, tempfile, urllib.request
from pathlib import Path


def main():
    p=argparse.ArgumentParser();p.add_argument('--output',type=Path,default=Path('features/public'));a=p.parse_args()
    root=Path(__file__).resolve().parents[1]
    manifest=json.loads((root/'data/manifests/feature_archive_20261108.json').read_text())
    if a.output.exists() and any(a.output.iterdir()):raise ValueError('Choose an empty feature directory')
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/'features.tar.gz';digest=hashlib.sha256()
        with urllib.request.urlopen(manifest['url']) as source,archive.open('wb') as target:
            while chunk:=source.read(1024*1024):target.write(chunk);digest.update(chunk)
        if digest.hexdigest()!=manifest['sha256']:raise ValueError('Archive checksum mismatch')
        a.output.mkdir(parents=True,exist_ok=True)
        expected=manifest['files']
        with tarfile.open(archive,'r:gz') as tar:
            members=tar.getmembers()
            if {m.name for m in members}!=set(expected) or len(members)!=len(expected):raise ValueError('Unexpected archive members')
            for member in members:
                if not member.isfile() or Path(member.name).name!=member.name:raise ValueError('Unsafe archive member')
                target=a.output/member.name
                with tar.extractfile(member) as inp,target.open('wb') as dest:shutil.copyfileobj(inp,dest)
                h=hashlib.file_digest(target.open('rb'),'sha256').hexdigest()
                if h!=expected[member.name]['sha256']:raise ValueError('Member checksum mismatch')
    print(f'Verified {manifest["n"]} proteins in {a.output}')


if __name__=='__main__':main()
```

## scripts/export_research_archive.py

```python
"""Maintainer export from the earlier research directory; no new model fitting.

Usage: python scripts/export_research_archive.py --archive PATH --feature-output PATH
The archive is not distributed. Public data, checksums and split membership are.
"""
import argparse, hashlib, json, shutil
from pathlib import Path
import numpy as np


def main():
    p=argparse.ArgumentParser();p.add_argument('--archive',type=Path,required=True);p.add_argument('--feature-output',type=Path,required=True);args=p.parse_args()
    src=args.archive;root=Path(__file__).resolve().parents[1];dest=args.feature_output;dest.mkdir(parents=True,exist_ok=True)
    master=json.loads((src/'data/processed/universal_benchmark_export_20261106.json').read_text())['records']
    lookup={r['benchmark_record_id']:r for r in master}
    token_index=np.load(src/'sequence_architectures_20261107/data/residue_index_20261107.npz',allow_pickle=False)
    token_values=np.load(src/'sequence_architectures_20261107/data/esm650_residues_20261107.npy',mmap_mode='r',allow_pickle=False)
    token_lookup={str(s):i for i,s in enumerate(token_index['ids'])}
    all_records=[];blocks=[];chunks=[];summary=[];provenance=[]
    for name,oldname in [('psi','psi'),('garcia','external_garcia'),('nesg','external_nesg'),('esol',None)]:
        if name=='esol':
            rows=json.loads((src/'additional_evaluation_20261107/data/esol_evaluation_records_20261107.json').read_text())
            features=np.load(src/'additional_evaluation_20261107/data/esol_features_20261107.npz',allow_pickle=False)
            index=np.load(src/'additional_evaluation_20261107/data/esol_residue_index_20261107.npz',allow_pickle=False)
            values=np.load(src/'additional_evaluation_20261107/data/esol_esm650_residues_20261107.npy',mmap_mode='r',allow_pickle=False)
            ilook={str(s):i for i,s in enumerate(index['ids'])}
        else:
            features=np.load(src/f'esm650_pooling_20261106/data/{oldname}_20261106.npz',allow_pickle=False)
            rows=[lookup[str(i)] for i in features['ids']];index=token_index;values=token_values;ilook=token_lookup
        records=[]
        for i,(raw,identifier,seq) in enumerate(zip(rows,features['ids'],features['sequences'])):
            identifier,seq=str(identifier),str(seq)
            label=raw['binary_insoluble'] if name=='esol' else int(features['y'][i])
            record=dict(record_id=identifier,sequence=seq,length_aa=len(seq),insoluble=label,
                group_id=raw['family40_group'] if name=='esol' else str(features['groups'][i]),
                dataset=name,role='train' if name=='psi' else 'external',
                solubility_fraction=raw.get('solubility_fraction'),source_record_id=raw['source_record_id'],
                source_publication=raw['source_publication'],source_url=raw['source_urls'],
                endpoint=raw['source_label_endpoint'],antibody_screen=raw['antibody_screen'],
                assay_construct_caveat=raw.get('assay_construct_caveat','Canonical/core sequence; assay context is incomplete'),
                prior_exposure='Previously analyzed; retrospective panel, not a new blinded test',creation_date='2026-11-08')
            assert 20<=len(seq)<=100 and (name=='esol' or label==1-int(raw['binary_soluble']))
            records.append(record)
            ri=ilook[identifier];lo,hi=index['offsets'][ri:ri+2]
            assert np.array_equal(index['sequences'][ri],seq) and hi-lo==len(seq)
            chunks.append((values,int(lo),int(hi)))
        file=root/f'data/public/{name}_records_20261108.json';file.write_text(json.dumps(records,indent=2)+'\n')
        all_records.extend(records);blocks.append({k:features[k] for k in ['ids','sequences','desc','esm650','esm650_max']})
        summary.append(dict(dataset=name,n=len(records),insoluble=sum(r['insoluble']==1 for r in records),soluble=sum(r['insoluble']==0 for r in records),unclassified=sum(r['insoluble'] is None for r in records),groups=len({r['group_id'] for r in records}),role=records[0]['role']))
        provenance.append(dict(dataset=name,json_sha256=hashlib.sha256(file.read_bytes()).hexdigest(),source_url=records[0]['source_url']))
    pooled={k:np.concatenate([b[k] for b in blocks]) for k in blocks[0]}
    pooled['offsets']=np.concatenate([[0],np.cumsum([hi-lo for _,lo,hi in chunks])]).astype(np.int64)
    np.savez_compressed(dest/'pooled_20261108.npz',**pooled)
    values=np.lib.format.open_memmap(dest/'residues_20261108.npy',mode='w+',dtype=np.float16,shape=(int(pooled['offsets'][-1]),1280))
    for i,(a,lo,hi) in enumerate(chunks):values[pooled['offsets'][i]:pooled['offsets'][i+1]]=a[lo:hi]
    values.flush();del values
    old=json.loads((src/'esm650_pooling_20261106/data/splits_20261106.json').read_text())
    split={'record_ids':blocks[0]['ids'].tolist(),'creation_date':'2026-11-08','positive_class':'insoluble',**old['psi']}
    (root/'data/splits/psi_five_seed_splits_20261108.json').write_text(json.dumps(split,indent=2)+'\n')
    (root/'data/manifests/dataset_summary_20261108.json').write_text(json.dumps(summary,indent=2)+'\n')
    (root/'data/manifests/provenance_20261108.json').write_text(json.dumps(provenance,indent=2)+'\n')
    from solubility_public_data.descriptors import sequence_features
    desc=sequence_features(pooled['sequences'])
    if not np.allclose(desc.to_numpy(),pooled['desc'],atol=1e-5,rtol=1e-5):raise ValueError('Descriptor definition mismatch')
    (root/'data/manifests/descriptor_names_20261108.json').write_text(json.dumps(desc.columns.tolist(),indent=2)+'\n')
    print(json.dumps(summary,indent=2))


if __name__=='__main__':main()
```

## scripts/extract_features.py

```python
"""Frozen pinned ESM2-650M embeddings; canonical residues only, no labels used."""
import argparse, json
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, EsmModel
from solubility_public_data.data import read_records
from solubility_public_data.descriptors import sequence_features

MODEL='facebook/esm2_t33_650M_UR50D'
REVISION='08e4846e537177426273712802403f7ba8261b6c'


def main():
    p=argparse.ArgumentParser();p.add_argument('--records',nargs='+',required=True);p.add_argument('--output',required=True);p.add_argument('--device',default='cpu');p.add_argument('--batch-size',type=int,default=8);a=p.parse_args()
    records=[r for path in a.records for r in read_records(path)]
    if len({r['record_id'] for r in records})!=len(records):raise ValueError('Duplicate IDs across inputs')
    out=Path(a.output);out.mkdir(parents=True,exist_ok=True)
    if any(out.iterdir()):raise ValueError('Use an empty feature directory')
    seq=[r['sequence'] for r in records];offsets=np.r_[0,np.cumsum(list(map(len,seq)))].astype(np.int64)
    tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REVISION)
    model=EsmModel.from_pretrained(MODEL,revision=REVISION).eval().to(a.device)
    tokens=np.lib.format.open_memmap(out/'residues_20261108.npy',mode='w+',dtype=np.float16,shape=(offsets[-1],1280))
    means=[];maxima=[]
    with torch.inference_mode():
        for start in range(0,len(seq),a.batch_size):
            ss=seq[start:start+a.batch_size]
            batch=tokenizer(ss,padding=True,return_tensors='pt',return_special_tokens_mask=True)
            special=batch.pop('special_tokens_mask').bool();valid=batch['attention_mask'].bool() & ~special
            hidden=model(**{k:v.to(a.device) for k,v in batch.items()}).last_hidden_state.cpu()
            for j,s in enumerate(ss):
                v=hidden[j][valid[j]].float().numpy();i=start+j
                if v.shape!=(len(s),1280):raise ValueError('Tokenizer residue mismatch')
                tokens[offsets[i]:offsets[i+1]]=v.astype(np.float16);means.append(v.mean(0));maxima.append(v.max(0))
            print(f'{min(start+a.batch_size,len(seq))}/{len(seq)}',flush=True)
    tokens.flush()
    np.savez_compressed(out/'pooled_20261108.npz',ids=np.array([r['record_id'] for r in records]),sequences=np.array(seq),desc=sequence_features(seq).to_numpy(),esm650=np.array(means),esm650_max=np.array(maxima),offsets=offsets)
    (out/'feature_metadata_20261108.json').write_text(json.dumps(dict(model=MODEL,revision=REVISION,labels_used=False,pooling='float32 mean/max of valid residues; token cache float16',n=len(seq)),indent=2)+'\n')


if __name__=='__main__':main()
```

## scripts/import_weak.py

```python
"""Local-only importer for the previously curated 17,638 independent weak rows.

Input is the old weak_20261106.npz. Nothing is downloaded, published or pushed.
Review the upstream terms before sharing inputs or derivative artifacts.
"""
import argparse,json
from pathlib import Path
import numpy as np


def main():
    p=argparse.ArgumentParser();p.add_argument('--input',required=True);p.add_argument('--output',default='private_data/weak_records_20261108.json');a=p.parse_args()
    z=np.load(a.input,allow_pickle=False)
    if len(z['ids'])!=17638 or int(z['y'].sum())!=6042:raise ValueError('Unexpected curated weak pool')
    records=[dict(record_id=str(i),sequence=str(s),insoluble=int(y),group_id=str(g),dataset='independent_weak',role='train',label_quality='weak mixed endpoint; review terms before redistribution') for i,s,y,g in zip(z['ids'],z['sequences'],z['y'],z['groups'])]
    out=Path(a.output);out.parent.mkdir(parents=True,exist_ok=True);out.write_text(json.dumps(records,indent=2)+'\n')
    from solubility_public_data.reporting import export_tables
    export_tables(out.with_suffix('.xlsx'),{'Records':records})
    print(f'Imported {len(records)} rows locally. Use extract_features.py; keep private_data ignored.')


if __name__=='__main__':main()
```

## scripts/smoke_validate.py

```python
"""Two-epoch software validation on 128 real PSI proteins; not a model study."""
import argparse,json,subprocess,sys
from pathlib import Path
import numpy as np
from solubility_public_data.data import read_records


def main():
    p=argparse.ArgumentParser();p.add_argument('--features',required=True);p.add_argument('--output',type=Path,default=Path('runs/smoke_20261108'));a=p.parse_args()
    root=Path(__file__).resolve().parents[1];records=read_records(root/'data/public/psi_records_20261108.json',True)
    rng=np.random.default_rng(90210);used=set();picked=[]
    for label in [1,0]:
        candidates=[r for r in records if r['insoluble']==label];count=0
        for i in rng.permutation(len(candidates)):
            r=candidates[i]
            if r['group_id'] in used:continue
            picked.append(r);used.add(r['group_id']);count+=1
            if count==64:break
        if count!=64:raise ValueError('Insufficient unique groups for smoke fixture')
    a.output.mkdir(parents=True,exist_ok=True);input_path=a.output/'smoke_records_20261108.json';input_path.write_text(json.dumps(picked,indent=2)+'\n')
    for architecture in ['residual_mlp','bilstm']:
        subprocess.run([sys.executable,'-m','solubility_public_data.cli','evaluate','--records',str(input_path),'--features',a.features,'--config',str(root/f'configs/{architecture}_public.json'),'--max-epochs','2','--json-only','--date','20261108','--external',*[str(root/f'data/public/{name}_records_20261108.json') for name in ['garcia','nesg','esol']],'--output',str(a.output/architecture)],check=True)


if __name__=='__main__':main()
```

## scripts/validate_data.py

```python
"""Validate release records, exact frozen splits, class polarity and XLSX parity."""
import hashlib,json
from pathlib import Path
from solubility_public_data.data import read_records
from solubility_public_data.evaluation import validate_splits


def main():
    root=Path(__file__).resolve().parents[1];allrows={};summary=[]
    for name,n,positive in [('psi',1686,368),('garcia',284,44),('nesg',152,32),('esol',198,24)]:
        path=root/f'data/public/{name}_records_20261108.json';rows=read_records(path,name=='psi')
        assert len(rows)==n and sum(r['insoluble']==1 for r in rows)==positive
        xlsx=read_records(path.with_suffix('.xlsx'),name=='psi')
        # Pandas can represent empty numeric fields as NaN; normalize first.
        for r in xlsx:
            if r.get('solubility_fraction')!=r.get('solubility_fraction'):r['solubility_fraction']=None
        assert rows==xlsx, f'XLSX/JSON mismatch: {name}'
        allrows[name]=rows;summary.append(dict(dataset=name,n=n,insoluble=positive,json_sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    psi=allrows['psi'];s=json.loads((root/'data/splits/psi_five_seed_splits_20261108.json').read_text())
    assert s['record_ids']==[r['record_id'] for r in psi]
    for mode in ['random','grouped']:validate_splits(psi,s[mode],mode)
    for name in ['garcia','nesg','esol']:
        assert not {r['sequence'] for r in psi}&{r['sequence'] for r in allrows[name]}
        assert not {r['group_id'] for r in psi}&{r['group_id'] for r in allrows[name]}
    print(json.dumps(dict(status='passed',rows=summary,distinct_seeds=5,oof_folds=3,external_training_family_overlap=0),indent=2))


if __name__=='__main__':main()
```

## tests/test_protocol.py

```python
import hashlib
import json
from pathlib import Path
import numpy as np
import pytest
import torch
from solubility_public_data.models import FusionModel, masked_pool
from solubility_public_data.data import FeatureStore, subset
from solubility_public_data.training import Estimator, inner_split
from solubility_public_data.evaluation import metrics, choose_threshold, make_splits, validate_splits


@pytest.fixture
def fixture(tmp_path):
    rng=np.random.default_rng(1);n=60;lengths=np.full(n,20);ids=np.array([f'test_{i}' for i in range(n)])
    seq=np.array(['ACDEFGHIKLMNPQRSTVW'+('A' if i%2 else 'C') for i in range(n)])
    np.savez(tmp_path/'pooled_20261108.npz',ids=ids,sequences=seq,desc=rng.normal(size=(n,202)).astype('f4'),esm650=rng.normal(size=(n,1280)).astype('f4'),esm650_max=rng.normal(size=(n,1280)).astype('f4'),offsets=np.r_[0,np.cumsum(lengths)])
    np.save(tmp_path/'residues_20261108.npy',rng.normal(size=(n*20,1280)).astype('f2'))
    # The fixture is synthetic and excluded from scientific metrics.
    s=dict(ids=ids,sequence_sha256=np.array([hashlib.sha256(s.encode()).hexdigest() for s in seq]),global_values=rng.normal(size=(n,4)),global_available=np.ones(n),residue_values=rng.normal(size=(n*20,3)),residue_offsets=np.arange(n+1)*20,residue_mask=np.ones(n*20,bool),residue_available=np.ones(n),patch_values=rng.normal(size=(n*2,5)),patch_offsets=np.arange(n+1)*2,patch_mask=np.ones(n*2,bool),patch_available=np.ones(n),quality=np.ones((n,3)),global_feature_names=np.array(['g0','g1','g2','g3']),residue_feature_names=np.array(['r0','r1','r2']),patch_feature_names=np.array(['p0','p1','p2','p3','p4']))
    np.savez(tmp_path/'structure.npz',**s)
    records=[dict(record_id=str(i),sequence=str(s),insoluble=j%2,group_id=f'group_{j}',role='train') for j,(i,s) in enumerate(zip(ids,seq))]
    return tmp_path,records


@pytest.mark.parametrize('architecture,expected',[('residual_mlp',244387),('bilstm',393284)])
def test_public_parameter_counts(architecture,expected):
    model=FusionModel(architecture)
    assert sum(p.numel() for p in model.parameters())==expected


@pytest.mark.parametrize('architecture',['residual_mlp','bilstm'])
@pytest.mark.parametrize('profile',['public','structure'])
def test_fit_reload_and_training_only_scalers(fixture,architecture,profile):
    root,records=fixture
    store=FeatureStore(root,root/'structure.npz' if profile=='structure' else None)
    data=store.align(records,architecture=='bilstm',profile=='structure')
    y=np.array([r['insoluble'] for r in records],np.float32);groups=np.array([r['group_id'] for r in records])
    config=dict(architecture=architecture,profile=profile,max_epochs=1,patience=1,batch_size=32)
    tr=np.arange(48);model=Estimator(config,store).fit(subset(data,tr),y[tr],groups[tr]);p=model.predict(data)
    assert np.isfinite(p).all() and np.all((p>=0)&(p<=1))
    assert model.metadata['inner_group_overlap']==0
    assert np.allclose(model.scalers['esm650']['mean'],data['esm650'][tr].mean(0),atol=1e-6)
    model.save(root/'model.pt',.4);loaded=Estimator.load(root/'model.pt',store)
    assert np.array_equal(p,loaded.predict(data)) and loaded.threshold==.4
    # A model checkpoint accepts a new feature store: it is not tied to training IDs.
    if profile == 'public':
        import os
        other=root/'new_candidates';other.mkdir()
        changed={k:v.copy() for k,v in store.pooled.items()}
        changed['ids']=np.array(['candidate_'+str(i) for i in range(len(records))])
        np.savez(other/'pooled_20261108.npz',**changed)
        os.link(root/'residues_20261108.npy',other/'residues_20261108.npy')
        new_records=[{**r,'record_id':str(i)} for r,i in zip(records,changed['ids'])]
        new_store=FeatureStore(other);new_model=Estimator.load(root/'model.pt',new_store)
        assert np.array_equal(p,new_model.predict(new_store.align(new_records,architecture=='bilstm')))
    assert model.metadata['n_training']==48


def test_bilstm_padding_invariance():
    torch.manual_seed(1);model=FusionModel('bilstm').eval()
    b={'desc':torch.randn(2,202),'esm650':torch.randn(2,1280),'esm650_max':torch.randn(2,1280),'residue':torch.randn(2,25,1280),'lengths':torch.tensor([20,25]),'mask':torch.arange(25)[None,:]<torch.tensor([20,25])[:,None]}
    with torch.no_grad():
        a=model(b)['logit'];b['residue']=torch.nn.functional.pad(b['residue'],(0,0,0,13),value=float('nan'));b['mask']=torch.nn.functional.pad(b['mask'],(0,13),value=False)
        assert torch.allclose(a,model(b)['logit'],atol=1e-7)


def test_patch_permutation_and_missing_structure(fixture):
    root,records=fixture;store=FeatureStore(root,root/'structure.npz');data=store.align(records,False,True)
    b=store.batch(data,np.arange(4),store.scalers(data,np.arange(48)))
    model=FusionModel(profile='structure',structure_dims=store.structure_dims).eval()
    with torch.no_grad():
        a=model(b)['logit'];b['structure_patch']=b['structure_patch'].flip(1);b['structure_patch_mask']=b['structure_patch_mask'].flip(1)
        assert torch.equal(a,model(b)['logit'])
        b['structure_available'].zero_();b['structure_residue_mask'].zero_();b['structure_patch_mask'].zero_();b['structure_quality'].zero_()
        result=model(b);assert torch.isfinite(result['logit']).all();assert torch.all(result['gates'][:,0]==0)


def test_threshold_polarity_and_empty_acceptance():
    y=np.array([0,0,1,1]);p=np.array([.1,.2,.7,.9]);t=choose_threshold(y,p,1)
    m=metrics(y,p,t);assert t==.7 and m['recall']==1 and m['soluble_retention']==1 and m['accepted_insoluble_fraction']==0
    assert metrics(y,p,0)['accepted_insoluble_fraction'] is None


def test_five_distinct_holdouts(fixture):
    _,records=fixture
    for mode in ['random','grouped']:
        s=make_splits(records,mode);assert len({tuple(sorted(v['test'])) for v in s.values()})==5


def test_id_sequence_guard(fixture):
    root,records=fixture;store=FeatureStore(root);records[0]['sequence']='A'*20
    with pytest.raises(ValueError,match='Sequence mismatch'):store.align(records)


def test_group_leakage_is_rejected(fixture):
    _,records=fixture;s=make_splits(records,'grouped')
    tr,te=s['11']['train'],s['11']['test']
    records[tr[0]]['group_id']=records[te[0]]['group_id']
    with pytest.raises(ValueError,match='Family leakage'):validate_splits(records,s,'grouped')


def test_inner_selection_does_not_receive_holdout_labels():
    y=np.array([0,1]*30);g=np.arange(60).astype(str)
    tr,va=inner_split(y[:48],g[:48],11);y[48:]=1-y[48:]
    tr2,va2=inner_split(y[:48],g[:48],11)
    assert np.array_equal(tr,tr2) and np.array_equal(va,va2)


def test_numeric_group_zero_is_valid(tmp_path):
    from solubility_public_data.data import read_records
    rows=[dict(record_id='a',sequence='A'*20,insoluble=0,group_id=0,role='train'),dict(record_id='b',sequence='C'*20,insoluble=1,group_id=1,role='train')]
    f=tmp_path/'records.json';f.write_text(json.dumps(rows))
    assert read_records(f,training=True)[0]['group_id']==0
```